# Diabetes Prediction: Basic Train vs. Test Comparison

Simplest possible version: one dataset, one split, three models trained in separate cells. No grid search, no class weighting, no scaling pipelines.

Goal: compare each model's metrics on the **training set** vs. the **test set**. A big gap (train much better than test) means overfitting. Both scores being low means underfitting.

Run the setup cells once, then run each model's cell independently (in any order).

In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from xgboost import XGBClassifier

RANDOM_STATE = 42

## Load data

In [2]:
df = pd.read_csv('diabetes_binary_health_indicators_BRFSS2015.csv')

X = df.drop(columns=['Diabetes_binary'])
y = df['Diabetes_binary']

df.shape

(253680, 22)

## Train/test split

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

X_train.shape, X_test.shape

((202944, 21), (50736, 21))

## Scoring helper

In [4]:
results = {}

def score(model, X, y):
    pred = model.predict(X)
    return {
        'accuracy': accuracy_score(y, pred),
        'precision': precision_score(y, pred),
        'recall': recall_score(y, pred),
        'f1': f1_score(y, pred),
    }

def train_and_score(name, model):
    model.fit(X_train, y_train)
    train_scores = score(model, X_train, y_train)
    test_scores = score(model, X_test, y_test)
    results[name] = {'train': train_scores, 'test': test_scores}

    print(f'{name}')
    print(f"  train -> {train_scores}")
    print(f"  test  -> {test_scores}")
    return model

## Logistic Regression

In [5]:
log_reg = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
log_reg = train_and_score('LogisticRegression', log_reg)

LogisticRegression
  train -> {'accuracy': 0.8638392857142857, 'precision': 0.5389547544156786, 'recall': 0.15754853768080065, 'f1': 0.24382234627698876}
  test  -> {'accuracy': 0.8621491643014821, 'precision': 0.5173530772790375, 'recall': 0.15815532607158014, 'f1': 0.24225352112676057}


## Random Forest

In [6]:
rf = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)
rf = train_and_score('RandomForest', rf)

RandomForest
  train -> {'accuracy': 0.994417179123305, 'precision': 0.9951477562933236, 'recall': 0.9646355695441525, 'f1': 0.9796541383087615}
  test  -> {'accuracy': 0.8594094922737306, 'precision': 0.4876256767208043, 'recall': 0.1783844956853869, 'f1': 0.26121180735370275}


## XGBoost

In [7]:
xgb = XGBClassifier(eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1)
xgb = train_and_score('XGBoost', xgb)

XGBoost
  train -> {'accuracy': 0.8765127325764743, 'precision': 0.6809995497523638, 'recall': 0.21395480425787744, 'f1': 0.3256101827184414}
  test  -> {'accuracy': 0.8633711762850835, 'precision': 0.5308976093820478, 'recall': 0.16650162682133257, 'f1': 0.25349989231100584}


## Comparison table

A large positive gap on accuracy/f1 signals overfitting; low scores on both signals underfitting.

In [8]:
rows = []
for name, sets in results.items():
    for set_name, metrics in sets.items():
        rows.append({'model': name, 'set': set_name, **metrics})

summary = pd.DataFrame(rows).round(3)

pivot = summary.pivot(index='model', columns='set', values=['accuracy', 'precision', 'recall', 'f1'])
for metric in ['accuracy', 'precision', 'recall', 'f1']:
    pivot[(metric, 'gap (train-test)')] = (pivot[(metric, 'train')] - pivot[(metric, 'test')]).round(3)
pivot = pivot.sort_index(axis=1, level=0)
pivot

accuracy                             f1         \
set                gap (train-test)   test  train gap (train-test)   test   
model                                                                       
LogisticRegression            0.002  0.862  0.864            0.002  0.242   
RandomForest                  0.135  0.859  0.994            0.719  0.261   
XGBoost                       0.014  0.863  0.877            0.073  0.253   

                                 precision                         recall  \
set                 train gap (train-test)   test  train gap (train-test)   
model                                                                       
LogisticRegression  0.244            0.022  0.517  0.539            0.000   
RandomForest        0.980            0.507  0.488  0.995            0.787   
XGBoost             0.326            0.150  0.531  0.681            0.047   

                                  
set                  test  train  
model                             
LogisticRegression  0.158  0.158  
RandomForest        0.178  0.965  
XGBoost             0.167  0.214